# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
preprocess_tree = ColumnTransformer([("numeric", SimpleImputer(strategy="median"), feature_columns)])
preprocess_scaled = ColumnTransformer([("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), feature_columns)])

**Respuesta:** Si escalamos antes de separar los folds, estaríamos calculando medias y desviaciones con datos que después se usan para validar. Eso contamina la evaluación. Dentro del pipeline, el escalador aprende solo con la parte de entrenamiento de cada fold.

## 3. Modelos candidatos

In [5]:
models = {
 "DecisionTree": Pipeline([("prep", preprocess_tree), ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
 "LogisticRegression": Pipeline([("prep", preprocess_scaled), ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
 "KNN": Pipeline([("prep", preprocess_scaled), ("model", KNeighborsClassifier())])
}

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
initial_comparison = []
for name, pipeline in models.items():
 scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
 initial_comparison.append({"model":name,"train_accuracy":scores["train_score"].mean(),"cv_accuracy":scores["test_score"].mean(),"cv_std":scores["test_score"].std()})
initial_comparison = pd.DataFrame(initial_comparison)
print(initial_comparison.to_string(index=False))

             model  train_accuracy  cv_accuracy   cv_std
      DecisionTree          1.0000     0.902217 0.058617
LogisticRegression          1.0000     0.979064 0.027713
               KNN          0.9771     0.950739 0.035274


## 5. Optimización de hiperparámetros

In [7]:
grids = {"DecisionTree":{"model__max_depth":[2,3,5,None],"model__min_samples_leaf":[1,2,4]},"LogisticRegression":{"model__C":[0.1,1,10]},"KNN":{"model__n_neighbors":[3,5,7,9],"model__weights":["uniform","distance"]}}
searches = {}
for name, pipeline in models.items():
 search = GridSearchCV(pipeline, grids[name], scoring="accuracy", cv=cv, n_jobs=1, refit=True)
 search.fit(X_train,y_train)
 searches[name] = search
tuned_comparison = pd.DataFrame([{"model":name,"cv_accuracy":search.best_score_,"best_params":str(search.best_params_)} for name,search in searches.items()]).sort_values("cv_accuracy",ascending=False)
print(tuned_comparison.to_string(index=False))

             model  cv_accuracy                                            best_params
LogisticRegression     0.979064                                        {'model__C': 1}
               KNN     0.957882 {'model__n_neighbors': 7, 'model__weights': 'uniform'}
      DecisionTree     0.902463  {'model__max_depth': 5, 'model__min_samples_leaf': 2}


## 6. Selección y persistencia del campeón

In [8]:
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")
metadata = {"champion":champion_name,"champion_name":champion_name,"cv_accuracy":float(searches[champion_name].best_score_),"best_params":searches[champion_name].best_params_}
(ARTIFACTS_DIR / "training_metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv",index=False)
print(metadata)

{'champion': 'LogisticRegression', 'champion_name': 'LogisticRegression', 'cv_accuracy': 0.9790640394088669, 'best_params': {'model__C': 1}}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:** Elegí el modelo con el mejor promedio de accuracy en validación cruzada, sin mirar el test. Comparé también los resultados de entrenamiento y la variación entre folds para detectar posibles señales de sobreajuste. Un árbol es fácil de explicar, mientras que regresión logística y KNN necesitan escalar las variables. La decisión final se basa en los números de CV y no en cuál modelo parece más sofisticado.

## Checklist
- [x] Procesamiento dentro de pipelines.
- [x] Tres modelos comparados.
- [x] Búsqueda de hiperparámetros con CV.
- [x] Selección sin usar test.